In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation, binary_fill_holes, binary_closing, label, binary_erosion
import os
import shutil
from google.colab import files

# === 1. PREPARA FOLDER SU GOOGLE DRIVE E UPLOAD FILE ===

my_folder = '/content/drive/MyDrive/Firme/'  # Cambia come vuoi!
os.makedirs(my_folder, exist_ok=True)
uploaded = files.upload()
nome_immagine = list(uploaded.keys())[0]
percorso_drive = os.path.join(my_folder, nome_immagine)
shutil.move(nome_immagine, percorso_drive)
print(f"File spostato in: {percorso_drive}")

# === FUNZIONI UTILI ===

def mostra_istogrammi(rgb_array, titolo):
    if rgb_array.shape[1] == 4:
        mask = rgb_array[:, 3] > 0
        data = rgb_array[mask, :3]
        note = "\n(solo pixel opachi)"
    else:
        data = rgb_array
        note = ""
    plt.figure(figsize=(12, 4))
    for i, colore in enumerate(['Red', 'Green', 'Blue']):
        plt.subplot(1, 3, i+1)
        plt.hist(data[:, i], bins=256, color=colore.lower(), alpha=0.7)
        plt.title(f'{colore} Channel - {titolo}{note}')
        plt.xlabel('Intensity')
        plt.ylabel('Count')
    plt.tight_layout()
    plt.show()

def convert_color_pixels(img_lettura, img_scrittura, mycolor):
    width, height = img_lettura.size
    for x in range(width):
        for y in range(height):
            r, g, b, a = img_lettura.getpixel((x, y))
            if a != 0:
                if (r, g, b) == mycolor:
                    all_white = True
                    for dx in [-1, 0, 1]:
                        for dy in [-1, 0, 1]:
                            if dx != 0 or dy != 0:
                                if 0 <= x + dx < width and 0 <= y + dy < height:
                                    r2, g2, b2, a2 = img_lettura.getpixel((x + dx, y + dy))
                                    if r2 != 255 or g2 != 255 or b2 != 255:
                                        all_white = False
                                        break
                    if all_white:
                        img_scrittura.putpixel((x, y), (255, 255, 255, 0))
    return img_scrittura

def convert_white_pixels(img_lettura, img_scrittura, mycolor):
    width, height = img_lettura.size
    for x in range(width):
        for y in range(height):
            r, g, b, a = img_lettura.getpixel((x, y))
            if a != 0:
                if (r, g, b) == (255, 255, 255):
                    all_mycolor = True
                    for dx in [-1, 0, 1]:
                        for dy in [-1, 0, 1]:
                            if dx != 0 or dy != 0:
                                if 0 <= x + dx < width and 0 <= y + dy < height:
                                    r2, g2, b2, a2 = img_lettura.getpixel((x + dx, y + dy))
                                    if (r2, g2, b2) != mycolor:
                                        all_mycolor = False
                                        break
                    if all_mycolor:
                        img_scrittura.putpixel((x, y), mycolor + (255,))
    return img_scrittura

# === 2. ELABORAZIONE IMMAGINE ===

try:
    img_lettura = Image.open(percorso_drive).convert("RGBA")
except Exception as e:
    print(f"Errore nell'apertura dell'immagine: {e}")
    exit()

width, height = img_lettura.size

# Istogramma immagine originale
rgb_originale = []
for x in range(width):
    for y in range(height):
        r, g, b, a = img_lettura.getpixel((x, y))
        rgb_originale.append((r, g, b, a))
rgb_originale_array = np.array(rgb_originale)
mostra_istogrammi(rgb_originale_array, "Original Image")

# Legenda colori
colori = {
    "Nero": (0, 0, 0),
    "Rosso": (255, 0, 0),
    "Verde": (0, 255, 0),
    "Blue": (0, 0, 255)
}
print("\nLegenda colori standard RGB:")
for nome, rgb in colori.items():
    print(f"{nome:<6} -> {rgb}")

# Parametri da utente
print("\nInserisci i valori soglia per ogni canale RGB (da 0 a 255):")
r_soglia = int(input("Soglia Red: "))
g_soglia = int(input("Soglia Green: "))
b_soglia = int(input("Soglia Blue: "))
print("Ora scegli il colore per la colorazione dei pixel (tratto):")
r_col = int(input("Red mycolor: "))
g_col = int(input("Green mycolor: "))
b_col = int(input("Blue mycolor: "))
mycolor = (r_col, g_col, b_col)

# Costruisci immagine modificata
img_scrittura = Image.new("RGBA", (width, height))
for z in range(10):
    for x in range(width):
        for y in range(height):
            r, g, b, a = img_lettura.getpixel((x, y))
            if a == 0:
                img_scrittura.putpixel((x, y), (255, 255, 255, 0))
            elif r < r_soglia or g < g_soglia or b < b_soglia:
                img_scrittura.putpixel((x, y), mycolor + (255,))
            else:
                img_scrittura.putpixel((x, y), (255, 255, 255, 0))

img_scrittura = convert_color_pixels(img_lettura, img_scrittura, mycolor)
img_scrittura = convert_white_pixels(img_lettura, img_scrittura, mycolor)

# Istogramma modificato
rgb_modificato = []
for x in range(width):
    for y in range(height):
        r, g, b, a = img_scrittura.getpixel((x, y))
        rgb_modificato.append((r, g, b, a))
rgb_modificato_array = np.array(rgb_modificato)
mostra_istogrammi(rgb_modificato_array, "Modified Image")

# Salva PNG in Drive
nome_file, _ = os.path.splitext(nome_immagine)
nome_immagine_modificata = nome_file + "_Mod.png"
img_scrittura.save(nome_immagine_modificata)
percorso_drive_mod = os.path.join(my_folder, nome_immagine_modificata)
shutil.move(nome_immagine_modificata, percorso_drive_mod)
print(f"Immagine modificata salvata come: {percorso_drive_mod}")

# Statistica tratto
arr = np.array(img_scrittura)
num_tratti_opachi = np.sum((arr[..., :3] == mycolor).all(axis=2) & (arr[..., 3] == 255))
num_tratti_trasp = np.sum((arr[..., :3] == mycolor).all(axis=2) & (arr[..., 3] == 0))
print(f"Pixel tratto visibile: {num_tratti_opachi}, Pixel tratto trasparente (dovrebbero essere 0): {num_tratti_trasp}")

# === 3. MODIFICA SPESSORE (ISPESSIRE o ASSOTTIGLIARE) ===

risp = input("Vuoi modificare lo spessore del tratto? (+% per ispessire, -% per assottigliare, n per nessuna modifica): ").strip()
if risp and risp != 'n':
    percent = float(risp)
    base_size = 3
    kernel_size = max(3, int(base_size * (1 + abs(percent) / 100.0)))
    if kernel_size % 2 == 0:
        kernel_size += 1  # meglio dispari

    print(f"Kernel: {kernel_size}x{kernel_size}")

    # Kernel circolare
    Y, X = np.ogrid[:kernel_size, :kernel_size]
    center = (kernel_size // 2, kernel_size // 2)
    distance = (X - center[1]) ** 2 + (Y - center[0]) ** 2
    radius = (kernel_size // 2) ** 2
    kernel = (distance <= radius).astype(np.uint8)

    # Maschera binaria: 1 = tratto, 0 = trasparenza
    pixels = np.array(img_scrittura)
    mask = (
        (pixels[..., 0] == mycolor[0]) &
        (pixels[..., 1] == mycolor[1]) &
        (pixels[..., 2] == mycolor[2]) &
        (pixels[..., 3] == 255)
    )

    if percent > 0:
        print("Ispessimento (dilatazione)...")
        mask_mod = binary_dilation(mask, structure=kernel)
        mask_mod = binary_closing(mask_mod, structure=kernel)
        # Riempimento selettivo buchi piccoli (2-4 pixel)
        holes = binary_fill_holes(mask_mod) & (~mask_mod)
        labeled, num = label(holes)
        holes_to_fill = np.zeros_like(mask_mod)
        for i in range(1, num + 1):
            area = np.sum(labeled == i)
            if 2 <= area <= 3:
                holes_to_fill |= (labeled == i)
        mask_mod = mask_mod | holes_to_fill
        suffix = "_thick"
    elif percent < 0:
        print("Assottigliamento (erosione)...")
        mask_mod = binary_erosion(mask, structure=kernel)
        suffix = "_thin"
    else:
        print("Nessuna modifica al tratto.")
        mask_mod = mask.copy()
        suffix = "_orig"

    # Crea nuova immagine RGBA
    img_mod_array = np.zeros_like(pixels)
    img_mod_array[..., 0][mask_mod] = mycolor[0]
    img_mod_array[..., 1][mask_mod] = mycolor[1]
    img_mod_array[..., 2][mask_mod] = mycolor[2]
    img_mod_array[..., 3][mask_mod] = 255

    img_mod = Image.fromarray(img_mod_array.astype(np.uint8), "RGBA")

    nome_mod = nome_file + suffix + ".png"
    img_mod.save(nome_mod)
    percorso_drive_mod2 = os.path.join(my_folder, nome_mod)
    shutil.move(nome_mod, percorso_drive_mod2)
    print(f"Immagine modificata salvata come: {percorso_drive_mod2}")

else:
    print("Nessuna modifica di spessore richiesta.")

print("\nFINE! Hai ora i file PNG con trasparenza (_Mod e opzionalmente _thick/_thin, con buchi piccoli riempiti solo se ispessisci) nella cartella Drive selezionata.")
